# ¹H NMR Analysis of Honey Samples
## Metabolomics Workshop — FBMFOR Food Fraud Practicals

This notebook analyses **500 MHz ¹H NMR spectra** from the food fraud practical.
It reads Bruker TopSpin data directly from the instrument folder structure,
displays the reference compound (sugar) spectra and the unknown sample spectra
as waterfall plots, then performs **PCA-based chemometric analysis** to explore
how the unknown samples relate to the known standards.

---

### Data layout

```
data/nmr/
  2026_03_09_metabolomicsWORKSHOP_Fructose/   ← reference compound folders
  2026_03_09_metabolomicsWORKSHOP_Furfural/
  2026_03_11_metabolomicsWORKSHOP_Glucose/
  2026_03_12_metabolomicsWORKSHOP_Sucrose/
  2026_03_13_metabolomicsWORKSHOP_Xylitol/
  2026_03_17_metabolomicsWORKSHOP_DATA/        ← unknown honey samples
    10/  20/  30/  40/  50/
```

Each experiment folder contains:
- `acqus`       — acquisition parameters (spectrometer frequency, sweep width, …)
- `pdata/1/1r`  — processed real spectrum (binary, 32-bit integers)
- `pdata/1/procs` — processing parameters (spectral width, offset, scaling, …)
- `pdata/1/title` — free-text title stored by the operator

---

### Running on Google Colab

The notebook fetches all data automatically from GitHub — no file upload needed.

1. Click **Runtime → Run all** (or press `Ctrl+F9`)
2. When prompted, allow the notebook to clone the repository
3. All figures are saved to `data/processed/` inside the cloned folder

To use **your own data**, replace the contents of `data/nmr/` with your Bruker
experiment folders before running Section 3 onwards, or update `NMR_DIR` in
Section 0 to point to your data location.

## 0 · Environment setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Detect whether we are running in Google Colab or locally, and set paths.
#
# LOCAL layout (notebook in <project>/notebooks/):
#   NMR data  →  <project>/data/nmr/
#   Figures   →  <project>/data/processed/
#
# COLAB layout (data fetched automatically from GitHub):
#   NMR data  →  /content/fbmfor-foodfraud/data/nmr/
#   Figures   →  /content/fbmfor-foodfraud/data/processed/
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

try:
    import google.colab      # noqa: F401  — only present inside Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # ── Download data from GitHub ─────────────────────────────────────────────
    # Clone the repository (shallow, main branch only) so all data files are
    # available without any manual upload.  The clone is skipped if it already
    # exists (e.g. if you re-run this cell).
    import subprocess, os
    repo_url = "https://github.com/ggkuhnle/fbmfor-foodfraud.git"
    clone_dir = Path("/content/fbmfor-foodfraud")
    if not clone_dir.exists():
        print("Cloning repository from GitHub …")
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, str(clone_dir)],
            check=True
        )
        print("Clone complete ✓")
    else:
        print("Repository already present, skipping clone.")

    NMR_DIR    = clone_dir / "data" / "nmr"
    OUTPUT_DIR = clone_dir / "data" / "processed"
    print("▶ Running on Google Colab")
else:
    # ── Local paths ───────────────────────────────────────────────────────────
    NMR_DIR    = Path('../data/nmr')
    OUTPUT_DIR = Path('../data/processed')
    print("▶ Running locally")

print(f"   NMR data  : {NMR_DIR.resolve()}")
print(f"   Output    : {OUTPUT_DIR.resolve()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory ready ✓")


## 1 · Imports

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# All packages are pre-installed in Google Colab and standard conda/pip
# scientific Python environments.
# ─────────────────────────────────────────────────────────────────────────────

import re                                # Parsing Bruker JCAMP-DX parameter files
import struct                            # (not used directly but good to have for reference)
import numpy as np                       # Numerical arrays
import pandas as pd                      # Tabular data (for the PCA data matrix)
import matplotlib.pyplot as plt          # All plotting
import matplotlib.cm as cm               # Colour maps
from sklearn.decomposition import PCA    # Principal Component Analysis

%matplotlib inline

# Consistent style across all figures
plt.rcParams.update({
    'figure.dpi'     : 120,
    'savefig.dpi'    : 300,
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'legend.fontsize': 9,
})

print("Imports OK ✓")

## 2 · Reading Bruker TopSpin data

### Bruker NMR folder structure

Bruker TopSpin saves each experiment as a numbered sub-folder containing:

| File | Contents |
|------|----------|
| `acqus` | Acquisition parameters (JCAMP-DX text format) |
| `pdata/1/procs` | Processing parameters (JCAMP-DX text format) |
| `pdata/1/1r` | Processed real spectrum — binary 32-bit signed integers |
| `pdata/1/title` | Free-text title entered by the operator |

### Converting the binary spectrum to ppm

The `1r` file is a flat array of `SI` 32-bit integers.
To recover calibrated intensities and a ppm axis we need from `procs`:

| Parameter | Meaning |
|-----------|---------|
| `SI` | Number of complex points in the processed spectrum |
| `SF` | Spectrometer frequency for the processed spectrum (MHz) |
| `SW_p` | Spectral width in **Hz** |
| `OFFSET` | Chemical shift of the **left** (highest ppm) edge |
| `NC_proc` | Intensity scaling exponent: intensity × 2^NC_proc |
| `BYTORDP` | Byte order: 0 = little-endian (Intel), 1 = big-endian |

ppm axis: `np.linspace(OFFSET, OFFSET − SW_p/SF, SI)`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper functions for reading Bruker TopSpin data
# ─────────────────────────────────────────────────────────────────────────────

def read_jcamp(path: Path) -> dict:
    """
    Parse a Bruker JCAMP-DX parameter file (acqus, procs, …).

    Each parameter line looks like:  ##$PARAMNAME= value
    Returns a dict mapping parameter name → value string.
    """
    params = {}
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            # Match lines starting with ##$ (Bruker private parameters)
            m = re.match(r'##\$([\w]+)=\s*(.*)', line.strip())
            if m:
                params[m.group(1)] = m.group(2).strip()
    return params


def read_bruker_1r(exp_dir: Path) -> tuple:
    """
    Read a processed Bruker 1D ¹H NMR spectrum from a TopSpin experiment folder.

    Parameters
    ----------
    exp_dir : Path
        Path to the numbered experiment folder (e.g. .../Fructose/12).

    Returns
    -------
    ppm  : np.ndarray, shape (SI,)
        Chemical shift axis in ppm, running from high to low field
        (largest ppm on the left, as per NMR convention).
    data : np.ndarray, shape (SI,)
        Intensity values in absolute units (after NC_proc scaling).
    meta : dict
        'title'  — operator-entered title string
        'name'   — experiment folder name
        'SF'     — spectrometer frequency in MHz
    """
    # ── Read processing parameters ────────────────────────────────────────────
    proc = read_jcamp(exp_dir / 'pdata' / '1' / 'procs')

    si     = int(proc['SI'])          # number of spectral points
    sf     = float(proc['SF'])        # spectrometer frequency (MHz)
    sw_p   = float(proc['SW_p'])      # spectral width in Hz
    offset = float(proc['OFFSET'])    # ppm of the left (highest ppm) edge
    nc     = int(proc['NC_proc'])     # intensity scaling exponent
    # BYTORDP = 0 → little-endian (x86/arm), 1 → big-endian
    endian = '<' if int(proc.get('BYTORDP', '0')) == 0 else '>'

    # ── Read binary spectrum ──────────────────────────────────────────────────
    one_r_path = exp_dir / 'pdata' / '1' / '1r'
    with open(one_r_path, 'rb') as f:
        raw = np.frombuffer(f.read(), dtype=endian + 'i4')
    # raw may be longer than SI if the file has padding; slice to SI points.
    # NC_proc is a power-of-2 scale factor applied during TopSpin processing.
    data = raw[:si].astype(np.float64) * (2.0 ** nc)

    # ── Build the ppm axis ────────────────────────────────────────────────────
    # OFFSET is the ppm of the first point (left edge).
    # The last point is at OFFSET - SW_p/SF ppm.
    sw_ppm = sw_p / sf
    ppm = np.linspace(offset, offset - sw_ppm, si)

    # ── Read the operator title ───────────────────────────────────────────────
    title_path = exp_dir / 'pdata' / '1' / 'title'
    try:
        title = title_path.read_text(errors='ignore').strip()
    except FileNotFoundError:
        title = exp_dir.name

    meta = {'title': title, 'name': exp_dir.name, 'SF': sf}
    return ppm, data, meta


def normalise_spectrum(data: np.ndarray, ppm: np.ndarray,
                       region: tuple = (0.5, 9.0),
                       exclude: tuple = (4.5, 5.0)) -> np.ndarray:
    """
    Total spectral area normalisation within a diagnostic ppm region.

    Divides the spectrum by the sum of absolute intensities in `region`,
    after masking out `exclude` (typically the residual water peak at ~4.7 ppm).
    This equalises overall concentration differences between samples so that
    shape comparisons (PCA) reflect composition rather than dilution.

    Parameters
    ----------
    data    : intensity array
    ppm     : ppm axis (same length as data)
    region  : (low_ppm, high_ppm) — range used for normalisation
    exclude : (low_ppm, high_ppm) — range to mask before summing

    Returns
    -------
    data / total_area  (same shape as data)
    """
    # Build a boolean mask: True for points inside 'region' but outside 'exclude'
    in_region  = (ppm >= region[0])  & (ppm <= region[1])
    not_water  = (ppm < exclude[0])  | (ppm > exclude[1])
    mask       = in_region & not_water

    total = np.abs(data[mask]).sum()
    if total == 0:
        return data   # avoid division by zero for an empty spectrum
    return data / total

print("Reader functions defined ✓")

## 3 · Load reference compound spectra

The five reference folders each contain a single processed ¹H NMR spectrum of a
pure compound dissolved in D₂O:

| Compound | Role in honey |
|----------|--------------|
| **Glucose** | Major monosaccharide (~30 % of honey) |
| **Fructose** | Major monosaccharide (~38 % of honey) |
| **Sucrose** | Disaccharide — elevated in adulterated or fresh honey |
| **Xylitol** | Sugar alcohol — not naturally in honey; adulterant marker |
| **Furfural** | Heat/processing marker — forms from sugars under thermal stress |

The folder names encode both the compound and the date of acquisition;
we extract the compound name from the last `_`-delimited token.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Locate all reference compound directories.
# Convention: folder names match *metabolomicsWORKSHOP_<Compound>
# and do NOT end in 'DATA' (which is reserved for the unknown samples).
# ─────────────────────────────────────────────────────────────────────────────

ref_dirs = sorted([
    d for d in NMR_DIR.iterdir()
    if d.is_dir()
    and 'metabolomicsWORKSHOP' in d.name
    and not d.name.endswith('DATA')
])

print(f"Found {len(ref_dirs)} reference folders:")
for d in ref_dirs:
    print(f"  {d.name}")

# ── Load each reference ───────────────────────────────────────────────────────
# Each reference folder contains exactly one numbered sub-experiment.
# We take the first (and only) numbered subdirectory as the experiment path.

references = {}   # compound_name → (ppm, intensity, meta)

for ref_dir in ref_dirs:
    # Extract compound name from folder name, e.g.
    #   '2026_03_09_metabolomicsWORKSHOP_Fructose' → 'Fructose'
    compound = ref_dir.name.split('_')[-1]

    # Find the experiment sub-folder (a directory whose name is a number)
    exp_subdirs = sorted([d for d in ref_dir.iterdir()
                          if d.is_dir() and d.name.isdigit()])
    if not exp_subdirs:
        print(f"  WARNING: no numbered subdir in {ref_dir.name}, skipping")
        continue
    exp_dir = exp_subdirs[0]   # take the first (only) experiment

    ppm, data, meta = read_bruker_1r(exp_dir)
    references[compound] = (ppm, data, meta)
    print(f"  {compound:<12}  exp={meta['name']}  "
          f"title={meta['title']!r}  "
          f"ppm {ppm[0]:.1f}→{ppm[-1]:.1f}  "
          f"max={data.max():.2e}")

## 4 · Waterfall plot of reference compounds

Each trace is offset vertically so the spectra do not overlap.  
The x-axis follows the **NMR convention**: high ppm (low field) on the left.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 1: Waterfall plot of reference compounds (pure sugars)
#
# We display the 0.5–10 ppm region, which contains all diagnostic sugar signals.
# The region below 0.5 ppm (TSP reference at 0 ppm) and above 10 ppm are
# excluded as they carry no useful sugar information.
# ─────────────────────────────────────────────────────────────────────────────

PPM_MIN, PPM_MAX = 0.5, 10.0

n_ref = len(references)
ref_colors = cm.tab10(np.linspace(0, 0.9, n_ref))

fig, ax = plt.subplots(figsize=(13, 8))

# offset_step: based on the tallest peak across all reference spectra so that
# no two adjacent traces overlap, regardless of intensity differences.
all_maxima = []
for compound, (ppm, data, meta) in references.items():
    mask = (ppm >= PPM_MIN) & (ppm <= PPM_MAX)
    all_maxima.append(data[mask].max())
offset_step = np.median(all_maxima) * 1.2

for i, ((compound, (ppm, data, meta)), col) in enumerate(
        zip(references.items(), ref_colors)):
    mask = (ppm >= PPM_MIN) & (ppm <= PPM_MAX)
    ax.plot(ppm[mask], data[mask] + i * offset_step,
            color=col, linewidth=0.9)
    # Label at the right (high-field) edge of each trace
    ax.text(PPM_MIN - 0.05,
            data[mask][-1] + i * offset_step,
            compound, fontsize=9, va='center', ha='right', color=col)

# Shade the residual water region — suppressed by the pulse sequence
ax.axvspan(4.5, 5.0, alpha=0.07, color='steelblue')
ymax = ax.get_ylim()[1]
ax.text(4.75, ymax * 0.88, 'HDO\n(suppressed)',
        fontsize=7, ha='center', color='steelblue', alpha=0.8)

ax.invert_xaxis()
ax.set_xlabel('Chemical shift (ppm)')
ax.set_ylabel('Intensity + offset (a.u.)')
ax.set_title('¹H NMR Reference Compounds — Waterfall (500 MHz, D₂O)')
ax.set_xlim(PPM_MAX, PPM_MIN)
ax.set_yticks([])

plt.tight_layout()
out_path = OUTPUT_DIR / 'nmr_references_waterfall.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")


## 5 · Load unknown sample spectra

The sample data folder contains five numbered experiments (10, 20, 30, 40, 50).
Each was a honey or honey-like solution measured at 298 K (25 °C).
The operator title is stored in `pdata/1/title`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Locate and load the unknown sample spectra.
# The DATA folder contains numbered sub-experiments (10, 20, 30, 40, 50).
# Names are assigned here so students can relate their results to real products.
# ─────────────────────────────────────────────────────────────────────────────

# Find the DATA folder (ends with 'DATA')
data_dirs = [d for d in NMR_DIR.iterdir()
             if d.is_dir() and d.name.endswith('DATA')]

if not data_dirs:
    raise FileNotFoundError(
        f"No folder ending in 'DATA' found in {NMR_DIR.resolve()}.\n"
        "Check the NMR_DIR path in Section 0."
    )

sample_root = data_dirs[0]
print(f"Sample data folder: {sample_root.name}")

# Mapping from experiment number → sample name.
# Adjust if the class measures additional samples.
SAMPLE_NAMES = {
    10: "Sample A",
    20: "Sample B",
    30: "Sample C",
    40: "Sample D",
    50: "Sample E",
}

# Collect numbered sub-experiment directories, sorted numerically
sample_exp_dirs = sorted(
    [d for d in sample_root.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name)
)

samples = {}   # label → (ppm, intensity, meta)

print(f"\nFound {len(sample_exp_dirs)} sample experiments:")
for exp_dir in sample_exp_dirs:
    ppm, data, meta = read_bruker_1r(exp_dir)
    # Use the explicit name map; fall back to the operator title or exp number
    exp_num = int(exp_dir.name)
    label = SAMPLE_NAMES.get(exp_num, meta['title'] or f"Sample {exp_dir.name}")
    samples[label] = (ppm, data, meta)
    print(f"  exp {exp_dir.name}: {label!r}  max={data.max():.2e}")


## 6 · Waterfall plot of unknown samples

By eye, all five samples show a similar pattern of broad sugar peaks between
3.4 and 5.3 ppm. Look for differences in:
- The relative height of the anomeric peak (~5.2 ppm) — indicates glucose/fructose ratio
- The presence of furfural peaks at 7.5–8.4 ppm — heat treatment marker
- Peaks at 3.5–3.9 ppm — xylitol / sugar alcohol region

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 2: Waterfall plot of the five unknown honey samples
# ─────────────────────────────────────────────────────────────────────────────

n_samples = len(samples)
sample_colors = cm.Set2(np.linspace(0, 0.85, n_samples))

fig, ax = plt.subplots(figsize=(13, 8))

all_maxima_s = []
for label, (ppm, data, meta) in samples.items():
    mask = (ppm >= PPM_MIN) & (ppm <= PPM_MAX)
    all_maxima_s.append(data[mask].max())
offset_step_s = np.median(all_maxima_s) * 1.2

for i, ((label, (ppm, data, meta)), col) in enumerate(
        zip(samples.items(), sample_colors)):
    mask = (ppm >= PPM_MIN) & (ppm <= PPM_MAX)
    ax.plot(ppm[mask], data[mask] + i * offset_step_s,
            color=col, linewidth=0.9)
    ax.text(PPM_MIN - 0.05,
            data[mask][-1] + i * offset_step_s,
            label, fontsize=9, va='center', ha='right', color=col)

ax.axvspan(4.5, 5.0, alpha=0.07, color='steelblue')
ymax = ax.get_ylim()[1]
ax.text(4.75, ymax * 0.88, 'HDO\n(suppressed)',
        fontsize=7, ha='center', color='steelblue', alpha=0.8)

ax.invert_xaxis()
ax.set_xlabel('Chemical shift (ppm)')
ax.set_ylabel('Intensity + offset (a.u.)')
ax.set_title('¹H NMR Honey Samples — Waterfall (500 MHz, D₂O)')
ax.set_xlim(PPM_MAX, PPM_MIN)
ax.set_yticks([])

plt.tight_layout()
out_path = OUTPUT_DIR / 'nmr_samples_waterfall.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")


## 7 · Estimated Glucose / Fructose ratio

Honey is predominantly fructose (~38 %) and glucose (~30 %).
The ratio between the two is characteristic of floral origin and is one of
the most discriminating parameters in honey authentication.

### Why use the anomeric peaks?

In D₂O solution, monosaccharides rapidly interconvert between their
α and β forms (mutarotation). The anomeric protons (H-1) are well separated
from the overlapping ring CH/CH₂ signals and provide relatively clean integrals:

| Peak | Chemical shift | Assignment |
|------|---------------|----------|
| α-Glc H-1 | ~5.22 ppm | α-glucopyranose anomeric proton |
| β-Glc H-1 | ~4.63 ppm | β-glucopyranose anomeric proton |
| H3/4      | ~4.00 ppm | β-fructopyranose ring protons |
| H5        | ~4.10 ppm | β-fructopyranose ring protons |

**Glucose integral** = sum of the two anomeric windows (5.22 and 4.63 ppm).
**Fructose integral** = sum of the two anomeric windows (4.1 and 4.0 ppm)

> **Note on region selection**: a wider fructose window (4.0–5.2 ppm) would
> capture additional fructose signal but also includes the β-glucose peak at
> 4.63 ppm, inflating the fructose estimate. Adjust `FRUCTOSE_WINDOW` below
> and re-run to explore the effect on the ratio.

The spectra are normalised to total spectral area before integration,
so the integrals reflect relative composition rather than concentration.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Glucose / Fructose ratio estimation from ¹H NMR integrals
#
# Integration windows — adjust these to explore how region choice affects the ratio.
# ─────────────────────────────────────────────────────────────────────────────
# Glucose: two narrow windows bracketing the two anomeric resonances

GLUCOSE_WINDOWS = [
    (5.18, 5.28),   # alpha-glucose anomeric signal (~5.22 ppm)
    (4.60, 4.68),   # beta-glucose anomeric signal  (~4.63 ppm)
]

# Fructose: two narrow windows centred on characteristic fructose signals
FRUCTOSE_WINDOWS = [
    (4.07, 4.13),   # ~4.10 ppm
    (3.97, 4.03),   # ~4.00 ppm
]


def region_integral(ppm, data, low, high):
    """
    Return the sum of normalised intensities between `low` and `high` ppm.

    Total-area normalisation is applied first so that integrals are more
    directly comparable between samples despite differences in overall
    signal intensity.
    """
    norm = normalise_spectrum(data, ppm)
    mask = (ppm >= low) & (ppm <= high)
    return float(norm[mask].sum())


def summed_integral(ppm, data, windows):
    """Return the sum of integrals across multiple ppm windows."""
    return sum(region_integral(ppm, data, low, high) for low, high in windows)


# ── Compute integrals for each sample ───────────────────────────────────────
results = []

for label, (ppm, data, meta) in samples.items():
    i_glc_522 = region_integral(ppm, data, *GLUCOSE_WINDOWS[0])
    i_glc_463 = region_integral(ppm, data, *GLUCOSE_WINDOWS[1])
    i_fru_410 = region_integral(ppm, data, *FRUCTOSE_WINDOWS[0])
    i_fru_400 = region_integral(ppm, data, *FRUCTOSE_WINDOWS[1])

    i_glc = i_glc_522 + i_glc_463
    i_fru = i_fru_410 + i_fru_400

    ratio = i_fru / i_glc if i_glc > 0 else float("nan")

    results.append({
        "Sample": label,
        "Glc (5.22 ppm)": i_glc_522,
        "Glc (4.63 ppm)": i_glc_463,
        "Total Glc": i_glc,
        "Fru (4.10 ppm)": i_fru_410,
        "Fru (4.00 ppm)": i_fru_400,
        "Total Fru": i_fru,
        "Fru/Glc ratio": ratio,
    })

df_ratio = pd.DataFrame(results).set_index("Sample")

print("Glucose / fructose integral estimates")
print("(integrals are normalised area units — relative comparison only)\n")
print(df_ratio.to_string(float_format="{:.4f}".format))


# ── Bar chart of Fru/Glc ratio ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(df_ratio))
colors_bar = cm.Set2(np.linspace(0, 0.85, len(df_ratio)))

ax.bar(
    x,
    df_ratio["Fru/Glc ratio"],
    color=colors_bar,
    edgecolor="k",
    linewidth=0.6
)

# Reference line at ratio = 1 (equal glucose and fructose)
ax.axhline(
    1.0,
    color="grey",
    linestyle="--",
    linewidth=0.8,
    label="Fru = Glc (ratio 1.0)"
)

ax.set_ylabel("Fructose / glucose ratio (NMR integrals)")
ax.set_title("Estimated Fructose/Glucose Ratio — ¹H NMR Honey Samples")
ax.set_xticks(x)
ax.set_xticklabels(df_ratio.index, rotation=20, ha="right")
ax.legend(fontsize=9)

plt.tight_layout()
out_path = OUTPUT_DIR / "nmr_fru_glc_ratio.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"\nSaved -> {out_path}")


## 7 · PCA — chemometric analysis

### Strategy

With 262 144 spectral points per spectrum, using the raw spectrum directly would
make the PCA matrix enormous and noisy. Instead we use **spectral binning**:

1. Select a diagnostic region (0.5–9.0 ppm), excluding the HDO peak (4.5–5.0 ppm)
2. Divide it into **0.04 ppm wide bins**
3. Sum (integrate) the intensity in each bin → one number per bin per spectrum
4. This gives ~200 bins — manageable for PCA and analogous to bucket integration
   in traditional NMR metabolomics

### Normalisation

We apply **total spectral area normalisation** (TSN) before binning:
divide each spectrum by the sum of absolute intensities in the diagnostic region.
This removes differences in overall concentration or shimming and lets PCA focus
on *composition* rather than *dilution*.

### PCA pre-processing

Mean-centre the binned matrix (subtract the column mean) before PCA.
No variance scaling — this preserves the relative importance of large peaks.

### Dataset

We include **all spectra** — references and samples together — so the scores plot
shows where the unknowns sit relative to the known pure compounds.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 · Spectral binning + normalisation  (samples only)
#
# PCA is run on the unknown honey samples only.  The reference compound spectra
# are pure standards at arbitrary concentrations; including them would dominate
# the variance and obscure differences between the real honey samples.
# ─────────────────────────────────────────────────────────────────────────────

BIN_WIDTH  = 0.04    # ppm per bin
DIAG_LOW   = 0.5     # lower ppm limit
DIAG_HIGH  = 9.0     # upper ppm limit
WATER_LOW  = 4.5     # HDO exclusion
WATER_HIGH = 5.0

bin_edges   = np.arange(DIAG_LOW, DIAG_HIGH + BIN_WIDTH, BIN_WIDTH)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

water_mask = (bin_centres >= WATER_LOW) & (bin_centres <= WATER_HIGH)
good_bins  = ~water_mask
print(f"Total bins: {len(bin_centres)}  |  "
      f"Water bins excluded: {water_mask.sum()}  |  "
      f"Bins used for PCA: {good_bins.sum()}")


def bin_spectrum(ppm: np.ndarray, data: np.ndarray) -> np.ndarray:
    """
    Total-area normalise then bin a spectrum onto the predefined grid.

    Normalisation removes concentration / shimming differences between samples
    so that PCA reflects composition rather than dilution.
    Binning reduces 262 k spectral points to ~200 integral values and makes
    the analysis robust to small inter-sample chemical shift variations.
    """
    norm_data = normalise_spectrum(data, ppm,
                                   region=(DIAG_LOW, DIAG_HIGH),
                                   exclude=(WATER_LOW, WATER_HIGH))
    bin_indices = np.digitize(ppm, bin_edges)
    n_bins = len(bin_edges) - 1
    binned = np.zeros(n_bins)
    for b in range(1, n_bins + 1):
        pts = (bin_indices == b)
        if pts.any():
            binned[b - 1] = norm_data[pts].sum()
    return binned


# ── Build the data matrix from samples only ───────────────────────────────────
sample_labels  = []
sample_binned  = []

for label, (ppm, data, meta) in samples.items():
    sample_labels.append(label)
    sample_binned.append(bin_spectrum(ppm, data))

X_pca  = np.array(sample_binned)[:, good_bins]
wn_pca = bin_centres[good_bins]

print(f"\nData matrix for PCA: {X_pca.shape}  "
      f"({X_pca.shape[0]} samples × {X_pca.shape[1]} bins)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run PCA on the honey samples
# ─────────────────────────────────────────────────────────────────────────────

X_mc = X_pca - X_pca.mean(axis=0)

n_components = min(len(sample_labels), 5)
pca    = PCA(n_components=n_components)
scores = pca.fit_transform(X_mc)
explained = pca.explained_variance_ratio_ * 100

print("Variance explained per PC:")
for i, ev in enumerate(explained):
    print(f"  PC{i+1}: {ev:5.1f}%  {'█' * int(ev/2)}")
print(f"  Cumulative: {explained.sum():.1f}%")


## 8 · PCA Scores plot

Reference compounds (pure sugars) are plotted as **triangles (▲)**;
unknown honey samples as **circles (●)**.

If an unknown sample plots close to one of the reference compounds,
its ¹H NMR spectrum resembles that pure compound.
Most real honey samples will fall between the glucose and fructose references
(honey is ~68% sugars, ~38% fructose and ~30% glucose).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 3: PCA Scores plot (PC1 vs PC2) — honey samples only
# ─────────────────────────────────────────────────────────────────────────────

colors = cm.Set2(np.linspace(0, 0.85, len(sample_labels)))

fig, ax = plt.subplots(figsize=(8, 6))

for i, (label, col) in enumerate(zip(sample_labels, colors)):
    ax.scatter(scores[i, 0], scores[i, 1],
               color=col, s=130, zorder=4, edgecolors='k', linewidths=0.6)
    ax.annotate(label, (scores[i, 0], scores[i, 1]),
                textcoords='offset points', xytext=(7, 4),
                fontsize=9, color=col)

ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.6, linestyle='--')
ax.set_xlabel(f'PC1  ({explained[0]:.1f} % variance explained)')
ax.set_ylabel(f'PC2  ({explained[1]:.1f} % variance explained)')
ax.set_title('PCA Scores — ¹H NMR Honey Samples (0.5–9.0 ppm, 0.04 ppm bins)')

plt.tight_layout()
out_path = OUTPUT_DIR / 'nmr_pca_scores.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")


## 9 · PCA Loadings plot

The loadings show **which chemical shift bins drive each principal component**.
Positive loading at a given ppm → high intensity there pushes a spectrum
towards the positive end of that PC axis; negative loading → the opposite.

Comparing the loading profile to the reference spectra (Figure 1) reveals
the chemical identity of the discriminating signals.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4: PCA Loadings (PC1 and PC2) plotted as pseudo-spectra
#
# A loading has one value per bin.  Plotted against chemical shift it shows
# which ppm regions drive sample separation:
#   Positive loading → high intensity here pushes a sample towards the + end of that PC
#   Negative loading → the opposite.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for pc_idx, ax in enumerate(axes):
    loading = pca.components_[pc_idx]

    ax.fill_between(wn_pca, loading, 0,
                    where=(loading >= 0), color=f'C{pc_idx}', alpha=0.45)
    ax.fill_between(wn_pca, loading, 0,
                    where=(loading < 0),  color=f'C{pc_idx}', alpha=0.15)
    ax.plot(wn_pca, loading, color=f'C{pc_idx}', linewidth=0.8)
    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.set_ylabel(f'PC{pc_idx+1} loading\n({explained[pc_idx]:.1f} %)')

# NMR convention: high ppm on the left.
# invert_xaxis() is called ONCE here (outside the loop) — calling it inside
# the loop with sharex=True would invert twice and cancel out.
axes[0].invert_xaxis()

axes[-1].set_xlabel('Chemical shift (ppm)')
fig.suptitle('PCA Loadings — ¹H NMR Honey Samples (0.5–9.0 ppm, 0.04 ppm bins)', fontsize=13)

plt.tight_layout()
out_path = OUTPUT_DIR / 'nmr_pca_loadings.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

## 10 · Summary

| Step | Choice | Rationale |
|------|--------|-----------|
| Spectral region | 0.5–9.0 ppm | Contains all diagnostic sugar and furfural signals |
| HDO exclusion | 4.5–5.0 ppm | Residual water peak — suppressed by pulse sequence but still variable |
| Normalisation | Total spectral area (TSN) | Removes concentration and shimming differences |
| Binning | 0.04 ppm bins | Reduces 262 k points → ~200 bins; robust to small chemical shift variations |
| PCA pre-processing | Mean-centring only | Preserves relative peak heights; autoscaling would amplify noise bins |
| Chemometrics | PCA — scores + loadings | Standard exploratory method for NMR metabolomics |

---

### Interpreting the scores plot

- Samples that **cluster with Glucose or Fructose** are sugar-rich and consistent with honey.
- Samples close to **Sucrose** may be fresh honey or adulterated with cane/beet syrup.
- Samples near **Furfural** have been **heat-treated or stored at high temperature** — furfural forms from hexose sugars by dehydration.
- Samples near **Xylitol** may be adulterated with sugar alcohol-sweetened syrups.

